# 08 - Otimizacao ONNX

Objetivo: validar a Etapa 4 do Tech Challenge comparando o modelo servido em `.pkl` com a versao exportada para ONNX Runtime.

Este notebook nao escolhe um novo modelo. Ele otimiza o artefato definido em `configs/model_config.yaml` (`tfidf_logreg`) porque esse e o modelo realmente servido pela API e preserva `predict_proba`.

In [1]:
import pandas as pd

from src.config import MODELS_DIR, load_config
from src.optimization.onnx import compare_artifacts, export_model

model_name = load_config()["serving"]["model"]
model_dir = MODELS_DIR / model_name
model_dir

WindowsPath('C:/Users/willi/OneDrive/Documentos/Estudos/Python/tc03-cloud-mlops/models/tfidf_logreg')

## Exportacao

A conversao remove `lowercase` e `strip_accents` do grafo ONNX. Essa normalizacao passa a ocorrer em Python antes da chamada ao ONNX Runtime, evitando incompatibilidades conhecidas do `skl2onnx` com `strip_accents="unicode"` e problemas de locale em imagens Linux slim.

In [2]:
metadata = export_model(model_dir)
metadata

{'created_at': '2026-09-15T01:13:32+00:00',
 'source_model': 'model.pkl',
 'onnx_model': 'model.onnx',
 'classes': ['atencao', 'normal', 'urgente'],
 'input_name': 'texto',
 'label_output': 'label',
 'probability_output': 'probabilities',
 'preprocessing': ['lowercase', 'strip_accents_unicode'],
 'target_opset': 17,
 'sklearn_model_size_mb': 1.226,
 'onnx_model_size_mb': 0.84}

## Equivalencia e Latencia

A comparacao usa o mesmo split de teste, o mesmo protocolo de warm-up e chamadas single-sample descrito em `docs/NOTEBOOKS.md`. Pequenas divergencias de classe podem ocorrer em exemplos exatamente na fronteira por diferenca numerica entre scikit-learn e ONNX Runtime; o limite aceito fica em `configs/model_config.yaml`.

In [3]:
comparison = compare_artifacts(model_dir)
comparison["mismatch_rate"], comparison["mismatches"], comparison["checked_predictions"]

(0.011275964391691394, 19, 1685)

In [4]:
pd.read_csv(model_dir / "latency_comparison.csv")

,runtime,size_mb,latency_p50_ms,latency_p95_ms,latency_p99_ms,latency_mean_ms,latency_n_calls
0,sklearn,1.226,1.4629,2.72896,3.497032,1.640259,1000
1,onnx,0.840,0.5779,1.08842,2.181457,0.642037,1000
